In [1]:
import cProfile
import pstats
import io
import time
import pennylane as qml
from pennylane import numpy as np
import json

In [2]:

import os, sys
sys.path.append('../')

from pqcqec.utils.constants import QUBITS_FOR_GATES, QISKIT_GATES, GATE_IS_DIRECTIONAL, PENNYLANE_GATES
from pqcqec.noise.simple_noise import PennylaneNoisyGates
# from pqcqec.simulate.simulate import run_circuit_with_noise_model
from pqcqec.circuits.modify import pennylane_state_embedding


# Pure Simulator Code

In [3]:

def simple_circuit_simulator(circuit_ops, input_state, num_qubits, x_noise, z_noise):
    """Runs a quantum circuit with a noise model using PennyLane and PyTorch."""
    qdevice = qml.device("default.qubit", wires=num_qubits)

    @qml.qnode(qdevice)
    def circuit(state):
        pennylane_state_embedding(state, num_qubits)
        for i, op in enumerate(circuit_ops):
            gate, wires, param = op
            PENNYLANE_GATES[gate](wires=wires)
            for wire in wires:
                qml.RX(x_noise[i], wires=[wire])
                qml.RZ(z_noise[i], wires=[wire])

        return qml.state()

    return circuit(input_state)

def simple_circuit_generator(circuit_ops, num_qubits, x_noise, z_noise):
    """Runs a quantum circuit with a noise model using PennyLane and PyTorch."""
    qdevice = qml.device("default.qubit", wires=num_qubits)

    @qml.qnode(qdevice)
    def circuit(state):
        pennylane_state_embedding(state, num_qubits)
        for i, op in enumerate(circuit_ops):
            gate, wires, param = op
            PENNYLANE_GATES[gate](wires=wires)
            for wire in wires:
                qml.RX(x_noise[i], wires=[wire])
                qml.RZ(z_noise[i], wires=[wire])

        return qml.state()

    return circuit


# Constants

In [4]:
PQC_GATES = ['rz', 'rx', 'rz']
DATA_PATH = '../nogit/circuit_tokens/no_uncomp/5q_500g_circuit_data/'
GOOD_DATA_PATH = DATA_PATH + 'per_seed_data/'
BAD_DATA_PATH = DATA_PATH + 'poor_fidelity/'
CONFIG_PATH = DATA_PATH + 'config.json'

with open(CONFIG_PATH, 'r') as f:
    CONFIG = json.load(f)


NUM_QUBITS = CONFIG.get("qubits", 3)[0]
NUM_GATES = CONFIG.get("gates", 4)[0] # Multiply by 2 for uncomp gates. 
GATE_BLOCKS = CONFIG.get("gate_blocks", 4)
VALID_GATES = CONFIG.get("gate_dist", QISKIT_GATES)
if VALID_GATES:
    VALID_GATES = list(VALID_GATES.keys())
else:
    VALID_GATES = ['x', 'z', 'h', 'cx', 'cz']

print(f"Using {NUM_QUBITS} qubits, {NUM_GATES} gates, {GATE_BLOCKS} gate blocks, {VALID_GATES} valid gates.")

NOISE_DIST = {"x_rad": 0.01, "z_rad": 0.01, "delta_x": 0, "delta_z": 0}

PAD_ID = 0
UNDIRECTED_GATES = [gate for gate in VALID_GATES if not GATE_IS_DIRECTIONAL.get(gate, False)]
print(UNDIRECTED_GATES)

TRAIN_SZ = 0.8
VAL_SZ = 0.1
TEST_SZ = 0.1
BATCH_SIZE = 64

LEARNING_RATE = 5e-6
WEIGHT_DECAY = 1e-3

MAX_CIRCUITS = 1000


Using 5 qubits, 500 gates, 4 gate blocks, ['x', 'z', 'h', 'cx', 'cz'] valid gates.
['x', 'z', 'h', 'cz']


# Load Data

In [5]:

good_data = []
poor_data = []

for i, filename in enumerate(os.listdir(GOOD_DATA_PATH)):
    if i > 10000:
        break
    with open(GOOD_DATA_PATH + filename, 'r') as f:
        token_dict = json.load(f)
        good_data.append(token_dict['base_circuit_tokens'])
        f.close()

# for filename in os.listdir(BAD_DATA_PATH):
#     with open(BAD_DATA_PATH + filename, 'r') as f:
#         token_dict = json.load(f)
#         poor_data.append((token_dict['base_circuit_tokens'], token_dict['pqc_params'], token_dict['fidelity']))
#         f.close()

print(f"Number of good data samples: {len(good_data)}")
# print(f"Number of poor data samples: {len(poor_data)}")

Number of good data samples: 10001


In [6]:

# --- Profiling Script ---
def profile_functions(data, input_states=None, x_noise=None, z_noise=None):
    """
    Sets up and profiles the performance of both functions for repeated calls.
    """

    num_qubits = NUM_QUBITS

    circuit_ops = data
    

    # --- Profile simple_circuit_simulator (The Slower Method) ---
    print("Profiling simple_circuit_simulator...")
    pr_simulator = cProfile.Profile()
    pr_simulator.enable()
    for circuit in circuit_ops:
        simple_circuit_simulator(circuit, input_states, num_qubits, x_noise, z_noise)
    pr_simulator.disable()
    
    # --- Profile simple_circuit_generator (The Faster Method) ---
    print("Profiling simple_circuit_generator...")
    pr_generator = cProfile.Profile()
    pr_generator.enable()
    # Step 1: Generate the circuit once
    circuit_callable = simple_circuit_generator(circuit_ops[0], num_qubits, x_noise, z_noise)
    # Step 2: Run the generated circuit multiple times
    circuit_callable(input_states)
    pr_generator.disable()

    # --- Print Profiling Results ---
    print("\n--- simple_circuit_simulator Profile (repeated creation & execution) ---")
    s_simulator = io.StringIO()
    sortby = 'cumulative'
    ps_simulator = pstats.Stats(pr_simulator, stream=s_simulator).sort_stats(sortby)
    ps_simulator.print_stats()
    print(s_simulator.getvalue())

    print("\n--- simple_circuit_generator Profile (single creation, repeated execution) ---")
    s_generator = io.StringIO()
    ps_generator = pstats.Stats(pr_generator, stream=s_generator).sort_stats(sortby)
    ps_generator.print_stats()
    print(s_generator.getvalue())



In [ ]:

input_states = np.zeros((10, 2**NUM_QUBITS))
input_states[:,0] = 1.0
x_noise = np.ones(NUM_GATES) * 0.01
z_noise = np.ones(NUM_GATES) * 0.01

# Run the profiling script
profile_functions(good_data[:MAX_CIRCUITS], input_states, x_noise, z_noise)

Profiling simple_circuit_simulator...
Profiling simple_circuit_generator...

--- simple_circuit_simulator Profile (repeated creation & execution) ---
         1320823268 function calls (1268736711 primitive calls) in 423.324 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
1001/1000    0.005    0.000  422.743    0.423 /var/folders/t2/g8j2h9dd5gj5c2454fz6jz980000gn/T/ipykernel_58280/3287883014.py:1(simple_circuit_simulator)
1001/1000    0.579    0.001  422.516    0.423 /Users/ashutoshtiwari/Desktop/Research-Code/pqc-qec/.venv/lib/python3.13/site-packages/pennylane/workflow/qnode.py:917(__call__)
     1000    0.063    0.000  422.514    0.423 /Users/ashutoshtiwari/Desktop/Research-Code/pqc-qec/.venv/lib/python3.13/site-packages/pennylane/workflow/qnode.py:887(_impl_call)
11000/6000    0.083    0.000  401.181    0.067 /Users/ashutoshtiwari/Desktop/Research-Code/pqc-qec/.venv/lib/python3.13/site-packages/pennylane/logging/decor

In [ ]:
# --- New Analysis Function ---
def analyze_profiler_output(profile_stats, num_runs, label):
    """
    Parses and reports key metrics from cProfile stats for a given run.

    Args:
        profile_stats (pstats.Stats): The Stats object from cProfile.
        num_runs (int): The number of times the function was executed.
        label (str): A descriptive label for the run ("Simulator" or "Generator").
    """
    total_time = profile_stats.total_tt
    
    # Extracting stats for key functions by name
    stats_dict = profile_stats.stats
    
    # Initialize timings for different phases
    total_compilation_time = 0
    total_execution_time = 0
    
    for (filename, lineno, funcname), stats in stats_dict.items():
        # Look for the time spent compiling the qnode
        if 'qnode' in funcname or 'QNode' in funcname or '_build_quantum_tape' in funcname:
            total_compilation_time += stats[2] # cumulative time
        
        # Look for the time spent executing the circuit (the call to the qnode)
        # We assume the time in the main function call is a good proxy for execution
        if funcname == 'simple_circuit_simulator' or funcname == '__call__':
            total_execution_time += stats[2] # cumulative time

    # Calculate percentages
    compilation_percent = (total_compilation_time / total_time) * 100 if total_time > 0 else 0
    execution_percent = (total_execution_time / total_time) * 100 if total_time > 0 else 0
    
    # Print a formatted report
    print(f"\n--- Performance Analysis for '{label}' ({num_runs} runs) ---")
    print(f"Total Execution Time: {total_time:.4f} seconds")
    print("-" * 40)
    print(f"Compilation Time:     {total_compilation_time:.4f} seconds ({compilation_percent:.2f}%)")
    print(f"Execution Time:       {total_execution_time:.4f} seconds ({execution_percent:.2f}%)")
    
    # You can also print the most time-consuming calls for deeper insights
    print("\nTop 5 Time-Consuming Functions:")
    profile_stats.print_stats(5)
